# TF32 — Fine-tune ViSoBERT với dữ liệu đã duyệt

Chọn Runtime → Change runtime type → T4 GPU. Chạy lần lượt các cell rồi upload đúng một file `finetune-input.zip` khi được yêu cầu. Dữ liệu gồm 255 câu train và 65 câu validation, chia theo group_id với seed 42.

Nhãn 80 câu bổ sung đã được người dùng xác nhận. Huấn luyện dùng nhãn nhị phân clean/toxic và điểm gộp offensive + hate của checkpoint ba nhãn. Kết quả đánh giá sau huấn luyện gồm validation và kiểm tra hồi quy trên 300 câu đã phân tích trước đó. Test mới độc lập được bổ sung ở lượt đánh giá tiếp theo.

In [ ]:
%pip install -q "transformers==4.51.0" "tokenizers==0.21.2" sentencepiece safetensors \
    "pydantic>=2.6,<3" pyyaml

In [ ]:
import hashlib
import io
import json
import random
import sys
import uuid
import zipfile
from copy import deepcopy
from pathlib import Path

from google.colab import files

uploaded = files.upload()
assert len(uploaded) == 1, "Chọn đúng một finetune-input.zip"
bundle_bytes = next(iter(uploaded.values()))
bundle_path = Path("/content/tf32-finetune-input-" + uuid.uuid4().hex[:8] + ".zip")
bundle_path.write_bytes(bundle_bytes)
sys.path.insert(0, str(bundle_path))

from app.services.summary.pii import mask_pii  # noqa: E402 - bundled source
from evals.moderation.calibrate_visobert import (  # noqa: E402 - bundled source
    MODEL,
    REVISION,
    measure,
    predict,
    validate_rows,
)
from evals.moderation.finetune_visobert import (  # noqa: E402 - bundled source
    SEED,
    score_predictions,
    select_validation_threshold,
)

with zipfile.ZipFile(io.BytesIO(bundle_bytes)) as archive:
    manifest = json.loads(archive.read("manifest.json"))
    datasets = {}
    for name in ("train.jsonl", "validation.jsonl"):
        raw = archive.read(name)
        assert hashlib.sha256(raw).hexdigest() == manifest["dataset_sha256"][name]
        datasets[name] = [json.loads(line) for line in raw.decode("utf-8").splitlines()]
    regression = json.loads(archive.read("regression-pack.json"))
    assert (
        hashlib.sha256(archive.read("regression-pack.json")).hexdigest()
        == manifest["regression_pack_sha256"]
    )
train_rows = datasets["train.jsonl"]
validation_rows = datasets["validation.jsonl"]
for rows in (train_rows, validation_rows):
    validate_rows(rows)
    assert all(r["label_status"] == "reviewed" for r in rows)
assert not {r["group_id"] for r in train_rows} & {r["group_id"] for r in validation_rows}
assert not {r["id"] for r in train_rows + validation_rows} & {r["id"] for r in regression["rows"]}
RUN_DIR = Path("/content/tf32-finetune-" + uuid.uuid4().hex[:8])
RUN_DIR.mkdir()
BEST_DIR = RUN_DIR / "best-model"
print(
    "Train",
    len(train_rows),
    "Validation",
    len(validation_rows),
    "Regression",
    len(regression["rows"]),
)
print("Output", RUN_DIR)

## Nạp checkpoint và kiểm tra hàm loss

Giữ head ba nhãn hiện có. Loss nhị phân dùng hai logit tương đương `clean` và `logsumexp(offensive, hate)`, giúp học trực tiếp từ nhãn đã duyệt. Các thành phần offensive/hate sau fine-tune chỉ được sử dụng thông qua tổng điểm toxic trong thí nghiệm này.

In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoModelForSequenceClassification, AutoTokenizer

random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device", device)
if device.type == "cpu":
    print("CPU sẽ chạy chậm. Có thể chọn T4 GPU trước khi chạy phần huấn luyện.")
tokenizer = AutoTokenizer.from_pretrained(MODEL, revision=REVISION, trust_remote_code=False)
model, loading = AutoModelForSequenceClassification.from_pretrained(
    MODEL,
    revision=REVISION,
    trust_remote_code=False,
    use_safetensors=True,
    output_loading_info=True,
)
assert not loading["missing_keys"] and not loading["mismatched_keys"], loading
assert model.config.num_labels == 3
model.to(device)


def binary_loss(logits, labels, reduction="mean"):
    binary_logits = torch.stack(
        (logits[:, 0], torch.logsumexp(logits[:, 1:], dim=-1)),
        dim=-1,
    )
    return F.cross_entropy(binary_logits, labels, reduction=reduction)


probe = torch.tensor([[0.3, -0.2, 0.7], [0.1, 0.9, -0.4]], requires_grad=True)
labels = torch.tensor([0, 1])
loss = binary_loss(probe, labels)
prob = probe.softmax(-1)
expected = -torch.log(torch.stack((prob[0, 0], prob[1, 1:].sum()))).mean()
assert torch.allclose(loss, expected, atol=1e-6)
loss.backward()
assert torch.isfinite(probe.grad).all()
extreme = torch.tensor([[10000.0, -10000.0, -10000.0]])
assert torch.isfinite(binary_loss(extreme, torch.tensor([1])))
print("Binary loss and finite-gradient checks passed")

In [ ]:
BATCH_SIZE = 8
MAX_LENGTH = 256
EPOCHS = 3
LEARNING_RATE = 1e-5


def encode(rows):
    return tokenizer(
        [mask_pii(row["text"]) for row in rows],
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    ).to(device)


def evaluate_rows(rows):
    model.eval()
    predictions = {}
    total_loss = 0.0
    with torch.inference_mode():
        for offset in range(0, len(rows), BATCH_SIZE):
            batch = rows[offset : offset + BATCH_SIZE]
            logits = model(**encode(batch)).logits
            targets = torch.tensor([int(r["toxic"]) for r in batch], device=device)
            total_loss += binary_loss(logits, targets, reduction="sum").item()
            probabilities = logits.softmax(dim=-1)
            assert torch.isfinite(probabilities).all()
            for row, (clean, offensive, hate) in zip(batch, probabilities.tolist(), strict=True):
                predictions[row["id"]] = {
                    "clean": clean,
                    "offensive": offensive,
                    "hate": hate,
                    "toxic": min(1.0, offensive + hate),
                }
    return predictions, total_loss / len(rows)


original_validation, best_loss = evaluate_rows(validation_rows)
original_regression, _ = evaluate_rows(regression["rows"])
max_original_drift = max(
    abs(
        original_regression[key]["toxic"]
        - (regression["scores"][key]["offensive"] + regression["scores"][key]["hate"])
    )
    for key in original_regression
)
assert max_original_drift < 0.01, "Model/preprocessing differs from the previous run"
best_epoch = 0
best_validation = deepcopy(original_validation)
model.config.tf32_task = "binary_toxicity_from_grouped_logits"
model.save_pretrained(BEST_DIR, safe_serialization=True)
tokenizer.save_pretrained(BEST_DIR)
history = [{"epoch": 0, "validation_loss": best_loss}]
print("Original validation loss", best_loss, "Max baseline drift", max_original_drift)

## Huấn luyện

Thử nghiệm dùng 3 epoch với learning rate 1e-5, batch 8 và gradient clipping 1,0. Checkpoint được chọn bằng validation loss. Epoch 0 là mốc ban đầu để tránh tự động chọn một epoch làm validation loss xấu hơn.

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
for epoch in range(1, EPOCHS + 1):
    model.train()
    shuffled = list(train_rows)
    random.Random(SEED + epoch).shuffle(shuffled)
    total_train_loss = 0.0
    for offset in range(0, len(shuffled), BATCH_SIZE):
        batch = shuffled[offset : offset + BATCH_SIZE]
        targets = torch.tensor([int(r["toxic"]) for r in batch], device=device)
        optimizer.zero_grad(set_to_none=True)
        loss = binary_loss(model(**encode(batch)).logits, targets)
        assert torch.isfinite(loss), "Non-finite training loss"
        loss.backward()
        norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        assert torch.isfinite(norm), "Non-finite gradient"
        optimizer.step()
        total_train_loss += loss.item() * len(batch)
    current_validation, validation_loss = evaluate_rows(validation_rows)
    history.append(
        {
            "epoch": epoch,
            "train_loss": total_train_loss / len(train_rows),
            "validation_loss": validation_loss,
        }
    )
    print(history[-1])
    if validation_loss < best_loss:
        best_loss = validation_loss
        best_epoch = epoch
        best_validation = deepcopy(current_validation)
        model.save_pretrained(BEST_DIR, safe_serialization=True)
print("Selected epoch", best_epoch, "Validation loss", best_loss)

## Khóa lựa chọn trên validation

Sau khi nạp lại checkpoint tốt nhất, kiểm tra điểm validation khớp checkpoint đã lưu. Ngưỡng được chọn trên validation với số false positive tối đa bằng model gốc tại ngưỡng 0,97. Khi hòa điểm, ưu tiên recall harassment/hate, F1 rồi số false positive thấp hơn. Ngưỡng 1,01 tắt nhánh ViSoBERT nếu các ngưỡng còn lại không đáp ứng budget.

In [ ]:
del optimizer, model
if device.type == "cuda":
    torch.cuda.empty_cache()
model = AutoModelForSequenceClassification.from_pretrained(
    BEST_DIR,
    trust_remote_code=False,
    use_safetensors=True,
).to(device)
candidate_validation, reloaded_loss = evaluate_rows(validation_rows)
assert (
    max(
        abs(candidate_validation[key]["toxic"] - best_validation[key]["toxic"])
        for key in best_validation
    )
    < 1e-5
)
selection = select_validation_threshold(
    validation_rows,
    original_validation,
    candidate_validation,
)
selected_threshold = selection["threshold"]
run_manifest = {
    "source_model": MODEL,
    "source_revision": REVISION,
    "input_manifest": manifest,
    "input_zip_sha256": hashlib.sha256(bundle_bytes).hexdigest(),
    "seed": SEED,
    "epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "batch_size": BATCH_SIZE,
    "max_length": MAX_LENGTH,
    "device": str(device),
    "torch_version": torch.__version__,
    "best_epoch": best_epoch,
    "checkpoint_selection": "minimum_validation_binary_nll_including_epoch_0",
    "binary_objective": "CE(clean_logit, logsumexp(offensive_logit,hate_logit))",
    "threshold_selection": selection,
    "regression_role": "previously_seen_300_examples",
}
(RUN_DIR / "run-manifest.json").write_text(
    json.dumps(run_manifest, ensure_ascii=False, indent=2, allow_nan=False),
    encoding="utf-8",
)
print("Frozen threshold", selected_threshold, "Selected epoch", best_epoch)
if best_epoch == 0:
    print("Các epoch huấn luyện chưa giảm validation loss; checkpoint được chọn là epoch 0.")

## So sánh validation và kiểm tra hồi quy

Bộ 300 câu đã dùng để phát hiện hai nhóm lỗi, vì vậy bảng regression đo mức thay đổi trên lỗi đã biết. Model và ngưỡng đã được chọn ở các bước validation phía trên. OpenAI/Core giữ cùng kết quả trong regression pack để so sánh trên cùng đầu vào.

In [ ]:
import pandas as pd
from IPython.display import display

validation_summary = [
    {"method": "Original @0.97", **score_predictions(validation_rows, original_validation, 0.97)},
    {"method": "Selected @0.97", **score_predictions(validation_rows, candidate_validation, 0.97)},
    {
        "method": "Selected @validation threshold",
        **score_predictions(
            validation_rows,
            candidate_validation,
            selected_threshold,
        ),
    },
]
candidate_regression, _ = evaluate_rows(regression["rows"])


def hybrid_flags(predictions, threshold):
    flags = {}
    for key, score in regression["scores"].items():
        flags[key] = (
            score["openai_flagged"]
            or bool(score["core_codes"])
            or predictions[key]["toxic"] >= threshold
        )
    return flags


variants = {
    "OpenAI + Core": predict(regression, {"mode": "baseline"}),
    "Original hybrid @0.97": hybrid_flags(original_regression, 0.97),
    "Selected hybrid @0.97": hybrid_flags(candidate_regression, 0.97),
    "Selected hybrid @validation threshold": hybrid_flags(candidate_regression, selected_threshold),
}
regression_summary = [
    {"method": name, **measure(regression["rows"], flags)} for name, flags in variants.items()
]
by_kind = {
    kind: {
        name: measure([r for r in regression["rows"] if r["kind"] == kind], flags)
        for name, flags in variants.items()
    }
    for kind in sorted({r["kind"] for r in regression["rows"]})
}
old_errors = [
    {
        "id": key,
        "original_toxic": original_regression[key]["toxic"],
        "selected_toxic": candidate_regression[key]["toxic"],
        "selected_flag_at_validation_threshold": variants["Selected hybrid @validation threshold"][
            key
        ],
    }
    for key in ("tf32-h1-slang-023", "tf32-h1-slang-019")
]
report = {
    "evaluation_role": "validation_and_seen_regression",
    "selected_epoch": best_epoch,
    "selected_threshold": selected_threshold,
    "validation": validation_summary,
    "regression": regression_summary,
    "regression_by_kind": by_kind,
    "previous_false_positives": old_errors,
    "history": history,
}
display(pd.DataFrame(validation_summary))
display(pd.DataFrame(regression_summary))
display(pd.DataFrame(old_errors))
(RUN_DIR / "report.json").write_text(
    json.dumps(report, ensure_ascii=False, indent=2, allow_nan=False),
    encoding="utf-8",
)
(RUN_DIR / "predictions.json").write_text(
    json.dumps(
        {
            "original_validation": original_validation,
            "selected_validation": candidate_validation,
            "original_regression": original_regression,
            "selected_regression": candidate_regression,
        },
        allow_nan=False,
    ),
    encoding="utf-8",
)

## Tải kết quả

Gửi lại file ZIP kết quả để đối chiếu checkpoint, validation và các lỗi hồi quy. Cell tiếp theo tải riêng trọng số model để dùng lại. Colab có thể xóa runtime khi ngắt phiên, nên lưu model nếu muốn giữ checkpoint.

In [ ]:
result_zip = RUN_DIR / "tf32-visobert-finetune-results.zip"
with zipfile.ZipFile(result_zip, "x", zipfile.ZIP_DEFLATED) as archive:
    for name in ("run-manifest.json", "report.json", "predictions.json"):
        archive.write(RUN_DIR / name, name)
    archive.writestr("input-manifest.json", json.dumps(manifest, ensure_ascii=False))
files.download(str(result_zip))

In [ ]:
model_zip = RUN_DIR / "tf32-visobert-finetuned-model.zip"
with zipfile.ZipFile(model_zip, "x", zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(BEST_DIR.rglob("*")):
        if path.is_file():
            archive.write(path, "best-model/" + path.relative_to(BEST_DIR).as_posix())
    archive.write(RUN_DIR / "run-manifest.json", "run-manifest.json")
files.download(str(model_zip))